<a href="https://colab.research.google.com/github/fasyabrhns/Tensorflow-in-Actions/blob/main/3_2_Creating_Input_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 3: Keras and Data Retrieval in TensorFlow 2

<table align="left">
    <td>
        <a target="_blank" href="https://colab.research.google.com/github/thushv89/manning_tf2_in_action/blob/master/Ch03-Keras-and-Data-Retrieval/3.2.Creating_Input_Pipelines.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
    </td>
</table>

## Importing necessary libraries and some setup

In [1]:
import os
import random
import numpy as np
import requests
from zipfile import ZipFile
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Conv2D, Flatten
from tensorflow.keras.models import Sequential
import pandas as pd
import tensorflow_datasets as tfds

def fix_random_seed(seed):
    try:
        np.random.seed(seed)
    except NameError:
        print("Warning: Numpy is not imported. Setting the seed for Numpy failed.")
    try:
        tf.random.set_seed(seed)
    except NameError:
        print("Warning: TensorFlow is not imported. Setting the seed for TensorFlow failed.")
    try:
        random.seed(seed)
    except NameError:
        print("Warning: random module is not imported. Setting the seed for random failed.")

# Fixing the random seed
fix_random_seed(4321)
print("TensorFlow version: {}".format(tf.__version__))

data_dir = 'data'

if not os.path.exists(data_dir):
    os.makedirs(data_dir)


TensorFlow version: 2.19.0


## Using the `tf.data` API to retrieve data

Here we will be using the `tf.data` API to feed a dataset containing images of flowers. The dataset has a folder containing the images and a CSV file listing filenames and their corresponding label as an integer. We will write a TensorFlow data pipeline that does the following.

* Extract filenames and classes from the CSV
* Read in the images from the extracted filenames and resize them to 64x64
* Convert the class labels to one-hot encoded vectors
* Combine the processed images and one-hot encoded vectors to a single dataset
* Finally, shuffle the data and output as batches

### Downloading the data
The dataset is available at https://www.kaggle.com/olgabelitskaya/flower-color-images/data .

You need to download the zip file available in this URL and place it in the `data` folder in the `Ch03-Keras-and-Data-Retrieval` folder. You **do not** need to extract it as the following code will do it for you.

In [9]:
import os
from zipfile import ZipFile

zip_files = [f for f in os.listdir('data') if f.endswith('.zip')]

if not zip_files:
    print("Tidak ada file zip. Dataset kemungkinan sudah diekstrak.")
else:
    with ZipFile(os.path.join('data', zip_files[0]), 'r') as z:
        z.extractall('data')
    print("Dataset berhasil diekstrak.")


Tidak ada file zip. Dataset kemungkinan sudah diekstrak.


## Creating a tf.data.Dataset

Here we are creating the `tf.data` pipeline that executes the above steps.

In [10]:
# Section 3.2
# Code listing 3.5

import tensorflow as tf
import os
import tensorflow.keras.backend as K

K.clear_session() # Making sure we are clearing out the TensorFlow graph

# Read the CSV file with TensorFlow
# The os.path.sep at the end is important for the get_image function
data_dir = os.path.join('data', 'flower_images', 'flower_images') + os.path.sep
assert os.path.exists(data_dir)
csv_ds = tf.data.experimental.CsvDataset(
    os.path.join(data_dir,'flower_labels.csv') , ("",-1), header=True
)
# Separate the image names and labels to two separate sets
fname_ds = csv_ds.map(lambda a,b: a)
label_ds = csv_ds.map(lambda a,b: b)

def get_image(file_path):

    img = tf.io.read_file(data_dir + file_path)
    # convert the compressed string to a 3D uint8 tensor
    img = tf.image.decode_png(img, channels=3)
    # Use `convert_image_dtype` to convert to floats in the [0,1] range.
    img = tf.image.convert_image_dtype(img, tf.float32)
    # resize the image to the desired size.
    return tf.image.resize(img, [64, 64])

# Get the images by running get_image across all the filenames
image_ds = fname_ds.map(get_image)
print("The image dataset contains: {}".format(image_ds))
# Create onehot encoded labels from label data
label_ds = label_ds.map(lambda x: tf.one_hot(x, depth=10))
# Zip the images and labels together
data_ds = tf.data.Dataset.zip((image_ds, label_ds))

# Shuffle the data so that we get a mix of labels in every batch
data_ds = data_ds.shuffle(buffer_size= 20)
# Define a batch of size 5
data_ds = data_ds.batch(5)


The image dataset contains: <_MapDataset element_spec=TensorSpec(shape=(64, 64, 3), dtype=tf.float32, name=None)>


In [11]:
# Iterate through the data to see what it contains
for item in data_ds:
    print(item)
    break

(<tf.Tensor: shape=(5, 64, 64, 3), dtype=float32, numpy=
array([[[[0.5852941 , 0.5088236 , 0.39411768],
         [0.5852941 , 0.50980395, 0.4009804 ],
         [0.5862745 , 0.51176476, 0.40490198],
         ...,
         [0.82156867, 0.7294118 , 0.62352943],
         [0.82745105, 0.74509805, 0.6392157 ],
         [0.8284314 , 0.75098044, 0.64509803]],

        [[0.59607846, 0.51862746, 0.40980396],
         [0.59411764, 0.5235294 , 0.40882355],
         [0.59607846, 0.52254903, 0.41470593],
         ...,
         [0.82941186, 0.73921573, 0.63725495],
         [0.8313726 , 0.7411765 , 0.64215684],
         [0.82745105, 0.7401961 , 0.63823533]],

        [[0.60882354, 0.5294118 , 0.41960788],
         [0.6127451 , 0.532353  , 0.42352945],
         [0.6117647 , 0.53431374, 0.42549023],
         ...,
         [0.8343138 , 0.7441176 , 0.63823533],
         [0.82450986, 0.7303922 , 0.6303922 ],
         [0.81274515, 0.71470594, 0.6147059 ]],

        ...,

        [[0.9196079 , 0.9117648 , 0

### Defining and training a model

Here we are defining a simple Convolution Neural Network (CNN) model to train it on the image data we just retrieved. You don't have to worry about the technical details of CNNs right now. We will discuss them in detail in the next chapter.

In [12]:
# Section 3.2

from tensorflow.keras.layers import Dense, Conv2D, Flatten
from tensorflow.keras.models import Sequential

# Defining a Convolution neural network for you to train for the flowers data
# We will discuss convolution neural networks in more detail later
model = Sequential([
    Conv2D(64,(5,5), activation='relu', input_shape=(64,64,3)),
    Flatten(),
    Dense(10, activation='softmax')
])

# Compiling the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['acc'])

# Training the model with the tf.data pipeline
model.fit(data_ds, epochs=10)

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


42/42 ━━━━━━━━━━━━━━━━━━━━ 4s 69ms/step - acc: 0.2016 - loss: 5.9168
Epoch 2/10
 2/42 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - acc: 0.2500 - loss: 1.8886

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - acc: 0.4529 - loss: 1.6728
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - acc: 0.5907 - loss: 1.0212
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 4s 84ms/step - acc: 0.8607 - loss: 0.3970
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - acc: 0.9830 - loss: 0.1674
Epoch 6/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 65ms/step - acc: 0.9975 - loss: 0.0552
Epoch 7/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - acc: 0.9967 - loss: 0.0417
Epoch 8/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 4s 84ms/step - acc: 0.9859 - loss: 0.0515
Epoch 9/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - acc: 0.9918 - loss: 0.0217
Epoch 10/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 77ms/step - acc: 1.0000 - loss: 0.0131


## Using Keras data generators to retrieve data

Instead of `tf.data` API let us use the Keras `ImageDataGenerator` to retrieve the data. As you can see, the `ImageDataGenerator` involves much less code than the using the `tf.data` API.

In [13]:
# Section 3.2
# Code listing 3.6

from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import pandas as pd

data_dir = os.path.join('data','flower_images', 'flower_images')

# Defining an image data generator provided in Keras
img_gen = ImageDataGenerator()

# Reading the CSV files containing filenames and labels
labels_df = pd.read_csv(os.path.join(data_dir, 'flower_labels.csv'), header=0)

# Generating data using the flow_from_dataframe function
gen_iter = img_gen.flow_from_dataframe(
    dataframe=labels_df, directory=data_dir, x_col='file', y_col='label', class_mode='raw', batch_size=5, target_size=(64,64))

# Iterating through the data
for item in gen_iter:
    print(item)
    break

Found 210 validated image filenames.
(array([[[[  0.,   0.,   0.],
         [  0.,   0.,   0.],
         [  0.,   0.,   0.],
         ...,
         [  0.,  74.,   0.],
         [ 15.,  60.,   0.],
         [ 65.,  37.,   0.]],

        [[ 11.,  11.,  11.],
         [ 11.,  11.,  11.],
         [ 25.,  23.,  20.],
         ...,
         [ 62., 109.,  50.],
         [ 63., 113.,  52.],
         [ 65.,  52.,  45.]],

        [[ 13.,  14.,  13.],
         [ 16.,  17.,  15.],
         [ 13.,  15.,  13.],
         ...,
         [ 61., 109.,  48.],
         [ 66., 114.,  52.],
         [ 71.,  86.,  51.]],

        ...,

        [[100.,  83.,  72.],
         [104.,  89.,  75.],
         [ 93.,  80.,  66.],
         ...,
         [ 89.,  83.,  71.],
         [ 94.,  81.,  67.],
         [ 14.,  17.,  18.]],

        [[ 92.,  80.,  64.],
         [115., 101.,  85.],
         [ 68.,  61.,  53.],
         ...,
         [ 80.,  71.,  57.],
         [ 75.,  67.,  56.],
         [ 56.,  49.,  44.]],

## Using the `tensorflow-datasets` library

Here we will use the `tensorflow-datasets` package. It is a curated list of popular datasets available for machine learning projects. With this package you can download a dataset in a single line. This means you don't have to worry about downloading/extracting/formatting data manually. All of that will be already done when you import data using the `tensorflow-datasets` library.

### Lists the available datasets

In [14]:
# Section 3.2

import tensorflow_datasets as tfds
import tensorflow as tf
# See all registered datasets
tfds.list_builders()

['abstract_reasoning',
 'accentdb',
 'aeslc',
 'aflw2k3d',
 'ag_news_subset',
 'ai2_arc',
 'ai2_arc_with_ir',
 'ai2dcaption',
 'aloha_mobile',
 'amazon_us_reviews',
 'anli',
 'answer_equivalence',
 'arc',
 'asimov_dilemmas_auto_val',
 'asimov_dilemmas_scifi_train',
 'asimov_dilemmas_scifi_val',
 'asimov_injury_val',
 'asimov_multimodal_auto_val',
 'asimov_multimodal_manual_val',
 'asqa',
 'asset',
 'assin2',
 'asu_table_top_converted_externally_to_rlds',
 'austin_buds_dataset_converted_externally_to_rlds',
 'austin_sailor_dataset_converted_externally_to_rlds',
 'austin_sirius_dataset_converted_externally_to_rlds',
 'bair_robot_pushing_small',
 'bc_z',
 'bccd',
 'beans',
 'bee_dataset',
 'beir',
 'berkeley_autolab_ur5',
 'berkeley_cable_routing',
 'berkeley_fanuc_manipulation',
 'berkeley_gnm_cory_hall',
 'berkeley_gnm_recon',
 'berkeley_gnm_sac_son',
 'berkeley_mvp_converted_externally_to_rlds',
 'berkeley_rpt_converted_externally_to_rlds',
 'big_patent',
 'bigearthnet',
 'billsum',
 '

### Download the Cifar10 dataset and view information

In [15]:
# Section 3.2

import tensorflow_datasets as tfds
import tensorflow.keras.backend as K

K.clear_session() # Making sure we are clearing out the TensorFlow graph

# Load a given dataset by name, along with the DatasetInfo
data, info = tfds.load("cifar10", with_info=True)
print(info)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cifar10/incomplete.0ELXHC_3.0.2/cifar10-train.tfrecord*...:   0%|         …

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cifar10/incomplete.0ELXHC_3.0.2/cifar10-test.tfrecord*...:   0%|          …

Dataset cifar10 downloaded and prepared to /root/tensorflow_datasets/cifar10/3.0.2. Subsequent calls will reuse this data.
tfds.core.DatasetInfo(
    name='cifar10',
    full_name='cifar10/3.0.2',
    description="""
    The CIFAR-10 dataset consists of 60000 32x32 colour images in 10 classes, with 6000 images per class. There are 50000 training images and 10000 test images.
    """,
    homepage='https://www.cs.toronto.edu/~kriz/cifar.html',
    data_dir='/root/tensorflow_datasets/cifar10/3.0.2',
    file_format=tfrecord,
    download_size=162.17 MiB,
    dataset_size=132.40 MiB,
    features=FeaturesDict({
        'id': Text(shape=(), dtype=string),
        'image': Image(shape=(32, 32, 3), dtype=uint8),
        'label': ClassLabel(shape=(), dtype=int64, num_classes=10),
    }),
    supervised_keys=('image', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'test': <SplitInfo num_examples=10000, num_shards=1>,
        'train': <SplitInfo nu

### Exploring the data

Here we will print the `data` and see what it provides. Then we will need to batch the data as data is provided as individual samples when you import it from `tensorflow-datasets`.

In [16]:
print(data)

{Split('train'): <_PrefetchDataset element_spec={'id': TensorSpec(shape=(), dtype=tf.string, name=None), 'image': TensorSpec(shape=(32, 32, 3), dtype=tf.uint8, name=None), 'label': TensorSpec(shape=(), dtype=tf.int64, name=None)}>, Split('test'): <_PrefetchDataset element_spec={'id': TensorSpec(shape=(), dtype=tf.string, name=None), 'image': TensorSpec(shape=(32, 32, 3), dtype=tf.uint8, name=None), 'label': TensorSpec(shape=(), dtype=tf.int64, name=None)}>}


In [17]:
# Print some training data
train_ds = data["train"].batch(16)
for item in train_ds:
    print(item)
    break

{'id': <tf.Tensor: shape=(16,), dtype=string, numpy=
array([b'train_16399', b'train_01680', b'train_47917', b'train_17307',
       b'train_27051', b'train_48736', b'train_26263', b'train_01456',
       b'train_19135', b'train_31598', b'train_12970', b'train_04223',
       b'train_27152', b'train_49635', b'train_04093', b'train_17537'],
      dtype=object)>, 'image': <tf.Tensor: shape=(16, 32, 32, 3), dtype=uint8, numpy=
array([[[[143,  96,  70],
         [141,  96,  72],
         [135,  93,  72],
         ...,
         [ 96,  37,  19],
         [105,  42,  18],
         [104,  38,  20]],

        [[128,  98,  92],
         [146, 118, 112],
         [170, 145, 138],
         ...,
         [108,  45,  26],
         [112,  44,  24],
         [112,  41,  22]],

        [[ 93,  69,  75],
         [118,  96, 101],
         [179, 160, 162],
         ...,
         [128,  68,  47],
         [125,  61,  42],
         [122,  59,  39]],

        ...,

        [[187, 150, 123],
         [184, 148, 

In [18]:
# Section 3.2

import tensorflow as tf

# Defining a dataset with batch size 16
train_ds = data["train"].batch(16)

# Creating a dataset that returns an (image, one-hot label) tuple
def format_data(x):
    return (x["image"], tf.one_hot(x["label"], depth=10))
train_ds = train_ds.map(format_data)

# Iterating the dataset
for item in train_ds:
    print(item)
    break

(<tf.Tensor: shape=(16, 32, 32, 3), dtype=uint8, numpy=
array([[[[143,  96,  70],
         [141,  96,  72],
         [135,  93,  72],
         ...,
         [ 96,  37,  19],
         [105,  42,  18],
         [104,  38,  20]],

        [[128,  98,  92],
         [146, 118, 112],
         [170, 145, 138],
         ...,
         [108,  45,  26],
         [112,  44,  24],
         [112,  41,  22]],

        [[ 93,  69,  75],
         [118,  96, 101],
         [179, 160, 162],
         ...,
         [128,  68,  47],
         [125,  61,  42],
         [122,  59,  39]],

        ...,

        [[187, 150, 123],
         [184, 148, 123],
         [179, 142, 121],
         ...,
         [198, 163, 132],
         [201, 166, 135],
         [207, 174, 143]],

        [[187, 150, 117],
         [181, 143, 115],
         [175, 136, 113],
         ...,
         [201, 164, 132],
         [205, 168, 135],
         [207, 171, 139]],

        [[195, 161, 126],
         [187, 153, 123],
         [186, 151

### Training a simple CNN on the Cifar10 data

In [19]:
# Section 3.2

from tensorflow.keras.layers import Dense, Conv2D, Flatten
from tensorflow.keras.models import Sequential

# Defining a simple convolution neural network to process the CIFAR data
model = Sequential([
    Conv2D(64,(5,5), activation='relu', input_shape=(32,32,3)),
    Flatten(),
    Dense(10, activation='softmax')
])
# Compiling the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['acc'])

# Fitting the model on the data for 25 epochs
model.fit(train_ds, epochs=25)

Epoch 1/25


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


3125/3125 ━━━━━━━━━━━━━━━━━━━━ 74s 24ms/step - acc: 0.1272 - loss: 11.0260
Epoch 2/25
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 66s 21ms/step - acc: 0.1493 - loss: 2.2668
Epoch 3/25
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 66s 21ms/step - acc: 0.1520 - loss: 2.2391
Epoch 4/25
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 65s 21ms/step - acc: 0.1533 - loss: 2.2123
Epoch 5/25
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 67s 21ms/step - acc: 0.1671 - loss: 2.1759
Epoch 6/25
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 65s 21ms/step - acc: 0.1845 - loss: 2.1377
Epoch 7/25
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 83s 21ms/step - acc: 0.1960 - loss: 2.1113
Epoch 8/25
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 65s 21ms/step - acc: 0.2060 - loss: 2.0791
Epoch 9/25
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 82s 21ms/step - acc: 0.2133 - loss: 2.0638
Epoch 10/25
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 65s 21ms/step - acc: 0.2214 - loss: 2.0478
Epoch 11/25
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 65s 21ms/step - acc: 0.2251 - loss: 2.0423
Epoch 12/25
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 65s 21ms/step - acc: 0.2337 - 